Json Parsing and Processing

In [2]:
import json 
import os 
dir="data/Json" 
os.makedirs("data/Json", exist_ok=True)

In [3]:
# Sample nested JSON data
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"}
            ]
        }
    ],
    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15
        }
    }
}

In [4]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

In [5]:
with open("data/Json/nested_data.json", "w") as f:
    json.dump(json_data, f, indent=4)

In [6]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]

with open("data/Json/jsonl_data.jsonl", "w") as f:
    for record in jsonl_data:
        json_line = json.dumps(record)
        f.write(json_line + "\n")

### Json Processing Strategies

In [7]:
from langchain_community.document_loaders import JSONLoader
import json

JSON LOADER

In [9]:
employee_loader = JSONLoader(file_path="data/Json/nested_data.json", jq_schema=".employees[]",text_content=False)
employees = employee_loader.load()
print(employees)
print(f"Loaded {len(employees)} employee records.")
for employee in employees:
    print(employee.page_content)
    print(employee.metadata)

[Document(metadata={'source': 'C:\\Users\\ArnavBhatia\\Desktop\\arnav\\udemy\\Rag-Krish naik\\DataIngestParsing\\data\\Json\\nested_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': 'C:\\Users\\ArnavBhatia\\Desktop\\arnav\\udemy\\Rag-Krish naik\\DataIngestParsing\\data\\Json\\nested_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]
Loaded 2 employee records.
{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data P

Custom Json 

In [10]:
from typing import List, Dict
from langchain_core.documents import Document
def process_json_intelligent(filepath:str)->List[Document]:
    with open(filepath, "r") as f:
        data = json.load(f)
    
    documents = []
    for emp in data.get("employees", []):
        content = f"""Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}

        Projects:"""
        for proj in emp.get('projects', []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"
        
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        documents.append(doc)
    return documents



In [11]:
json_documents = process_json_intelligent("data/Json/nested_data.json")
print(f"Processed {len(json_documents)} documents from JSON.")
for doc in json_documents:
    print(doc.page_content)
    print(doc.metadata)
    

Processed 2 documents from JSON.
Employee Profile:
        Name: John Doe
        Role: Software Engineer
        Skills: Python, JavaScript, React

        Projects:
- RAG System (Status: In Progress)
- Data Pipeline (Status: Completed)
{'source': 'data/Json/nested_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}
Employee Profile:
        Name: Jane Smith
        Role: Data Scientist
        Skills: Python, Machine Learning, SQL

        Projects:
- ML Model (Status: In Progress)
- Analytics Dashboard (Status: Planning)
{'source': 'data/Json/nested_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}
